In [ ]:
ROOT = "PersistenceUniversality/pd_example_files/"

alpha = 0.05

In [ ]:
import numpy as np
import mtd
import matplotlib.pyplot as plt

import scipy.stats as st
import statsmodels.api as sm
import statsmodels.stats.multitest as mt
# Universal distribtion
# -----------------------
RVSTD = st.gumbel_l()
# -----------------------


# from nature paper about noise
def slope(b,d,alpha,A):
    M = len(b)
    P = d/b
    P=P[P>1]
    P=np.log(np.log(P))
    B=RVSTD.mean() - P.mean()*A
    print(f'B: {B}')
    print(f'M: {M}')
    return np.exp((np.exp(-B)*np.log(M/alpha))**(1/A))


# the weird transformation
def process(P,A=1):
    P=P[P>1]
    P=np.log(np.log(P))
    rvtmp = st.loggamma(c=1,loc=0,scale=1)
    B=RVSTD.mean() - P.mean() * A
    P = A * P + B
    return P


# test significance 
def test(X, alpha):
    dist = RVSTD
    tmp = -X
    tmp.sort() 
    X = -tmp 
    pvals = 1 - dist.cdf(X) 
    res, pv, _, _ = mt.multipletests(pvals, alpha=alpha, method='bonferroni')
    return res, pv, pvals

---

### Let's try to reproduce base experiments from the paper

In [ ]:
import seaborn as sns
DPI = 300
FN_TITLE = 30
FN_AXES = 24
FN_LEGEND = 12

sns.set_theme(style="darkgrid")
FIG_SIZE = (18,5)
sns.set_context("talk")

def TwoPDplotLineLog(b,d,b1,d1,label,label1,alpha,ax):
    mx=max(np.max(d),np.max(d1))
    mx = np.log(1.1*mx)
    mn1 = min(np.min(b),np.min(b1))
    mn = np.log(mn1)
    ax.fill([mn,mx,mx],[mn,mn,mx],facecolor='black',edgecolor='gray',alpha=0.1)
  
    ax.scatter(np.log(b),np.log(d),facecolor='black', edgecolor='white',s=200, label=label,alpha=0.799*d/np.max(d)+0.2)
    ax.scatter(np.log(b1),np.log(d1), edgecolor='black',facecolor='white',s=200, label=label1,alpha=0.799*d1/np.max(d1)+0.2)
    
    m = np.log(slope(b,d,alpha,1))
    ax.plot([mn,mx],[mn+m,mx+m],':',c='blue')
    m = np.log(slope(b1,d1,alpha,1))
    ax.plot([mn,mx],[mn+m,mx+m],'--',c='blue')
    
    ax.set_xlabel('Log(Birth)',fontsize=32)
    ax.set_ylabel('Log(Death)',fontsize=32)
    leg=ax.legend(fontsize=FN_AXES, loc='lower right')
 #   for lh in leg.legendHandles: 
 #       lh.set_alpha(1)
    #ax.set_title('Persistence Diagram', fontsize=FN_TITLE)
    ax.set(xlim=(mn, mx), ylim=(mn, mx))
    ax.tick_params(axis='x', labelsize=24)
    ax.tick_params(axis='y', labelsize=24)

    ax.set_aspect('equal')

In [ ]:
RA = 0.4
RB = 0.2

# load files - these are stored points
# because we use Julia to show the
# corresponding cycles below.
# ---------------------------------
P = np.loadtxt(f'{ROOT}annulus.csv')
P1 = np.loadtxt(f'{ROOT}annulus1.csv')

# # compute persistence diagrams
barc = mtd.calc_cross_barcodes(P, P, batch_size1 = 5000, batch_size2 = 0, is_plot = True, pdist_device = "cuda")

barc1 = mtd.calc_cross_barcodes(P1, P1, batch_size1 = 5000, batch_size2 = 0, is_plot = True, pdist_device = "cuda")

# #plot persistence diagram
# plt.figure(figsize=(10,10))
# ax = plt.axes()
# TwoPDplotLineLog(b,d,b1,d1,f'R = {RA}',f'R = {RB}',0.05,ax)
# plt.savefig('annulus_PD.pdf', dpi=300,bbox_inches = "tight",pad_inches=0.1)

In [ ]:
d = barc[1][:,1]
b = barc[1][:,0]

d1 = barc1[1][:,1]
b1 = barc1[1][:,0]

plt.figure(figsize=(10,10))
ax = plt.axes()
TwoPDplotLineLog(b,d,b1,d1,f'R = {RA}',f'R = {RB}',0.05,ax)

In [ ]:
plt.figure(figsize = (8,8))
plt.scatter(P[:,0],P[:,1],label = f'R = {RA}')
plt.scatter(P1[:,0],P1[:,1],label = f'R = {RB}')
plt.legend()
plt.show()

### As we can see results with our code of barcodes generation totaly match with paper results

---

### Now we want to shift one annulus relative to another and to monitor the change of number of significant features in crossbarcodes

In [ ]:
np.random.seed(7)
for shift in np.arange(0,4,0.2):
    fig, (ax1, ax2) = plt.subplots(1, 2,figsize=(16,8))

    fig.suptitle(f"shift between annulus = {shift:.2f}")
    
    P = np.loadtxt(f'{ROOT}annulus.csv')
    Q = np.loadtxt(f'{ROOT}annulus1.csv') + np.array([shift,0])
    
    ax1.scatter(P[:,0],P[:,1],label = f'R = {RA}')
    ax1.scatter(Q[:,0],Q[:,1],label = f'R = {RB}')
    
    barc = mtd.calc_cross_barcodes(P, Q, batch_size1 = 1000, batch_size2 = 1000, is_plot = False, pdist_device = "cuda")

    d = barc[1][:,1]
    b = barc[1][:,0]
    
    mx=np.max(d)
    mx = np.log(1.1*mx)
    mn1 = np.min(b)
    mn = np.log(mn1)
    ax2.fill([mn,mx,mx],[mn,mn,mx],facecolor='black',edgecolor='gray',alpha=0.1)
    
    ax2.scatter(np.log(b),np.log(d),facecolor='black', edgecolor='white',s=50, alpha=0.6*d/np.max(d)+0.2)
    
    m = np.log(slope(b,d,alpha,1))
    ax2.plot([mn,mx],[mn+m,mx+m],':',c='blue')
    
    
    ax2.set_xlabel('Log(Birth)',fontsize=20)
    ax2.set_ylabel('Log(Death)',fontsize=20)
    ax2.set(xlim=(mn, mx), ylim=(mn, mx))
    ax2.tick_params(axis='x', labelsize=15)
    ax2.tick_params(axis='y', labelsize=15)
    ax2.set_title("H1 with noise")
    
    ax2.set_aspect('equal')
    plt.plot()

---

In [ ]:
np.random.seed(7)

P = np.loadtxt(f'{ROOT}annulus.csv')
Q = np.loadtxt(f'{ROOT}annulus1.csv') + np.array([2,0])

In [ ]:
plt.scatter(P[:,0],P[:,1])
plt.scatter(Q[:,0],Q[:,1])

In [ ]:
barc = mtd.calc_cross_barcodes(P, Q, batch_size1 = 100, batch_size2 = 1000, is_plot = True)

In [ ]:
plt.figure(figsize=(10,10))
ax = plt.axes()


d = barc[1][:,1]
b = barc[1][:,0]

mx=np.max(d)
mx = np.log(1.1*mx)
mn1 = np.min(b)
mn = np.log(mn1)
ax.fill([mn,mx,mx],[mn,mn,mx],facecolor='black',edgecolor='gray',alpha=0.1)

plt.scatter(np.log(b),np.log(d),facecolor='black', edgecolor='white',s=50, alpha=0.6*d/np.max(d)+0.2)


ax.set_xlabel('Log(Birth)',fontsize=20)
ax.set_ylabel('Log(Death)',fontsize=20)
ax.set(xlim=(mn, mx), ylim=(mn, mx))
ax.tick_params(axis='x', labelsize=15)
ax.tick_params(axis='y', labelsize=15)
ax.set_title("H1 with noise")
plt.plot()

### Let's find noisy points

In [ ]:
plt.figure(figsize=(10,10))
ax = plt.axes()


d = barc[1][:,1]
b = barc[1][:,0]

mx=np.max(d)
mx = np.log(1.1*mx)
mn1 = np.min(b)
mn = np.log(mn1)
ax.fill([mn,mx,mx],[mn,mn,mx],facecolor='black',edgecolor='gray',alpha=0.1)

plt.scatter(np.log(b),np.log(d),facecolor='black', edgecolor='white',s=50, alpha=0.6*d/np.max(d)+0.2)

m = np.log(slope(b,d,alpha,1))
ax.plot([mn,mx],[mn+m,mx+m],':',c='blue')


ax.set_xlabel('Log(Birth)',fontsize=20)
ax.set_ylabel('Log(Death)',fontsize=20)
ax.set(xlim=(mn, mx), ylim=(mn, mx))
ax.tick_params(axis='x', labelsize=15)
ax.tick_params(axis='y', labelsize=15)
ax.set_title("H1 with noise")

ax.set_aspect('equal')
plt.plot()

In [ ]:
P = d/b
X = process(P)
test(X, alpha)

all points are noisy

In [ ]:
import sys
sys.path.append("PersistenceUniversality/modules")
import vhdb as vhdb

# Generator
FL_ALL = vhdb.get_flist()

# Return patches from the VH database. - truncation
def patches(D,N,FL=[],RVAR = 0.2, RKNN = 0.3, K=15):
    P,I = vhdb.gen_all_patches(N, FL, RKNN, RVAR, K)
    return P[:,:D], I

In [ ]:
N = 10000
D = 8
Patch,I = patches(D,N, FL=[], RKNN=0.3, RVAR = 0.2, K=15)

In [ ]:
S = 5
P =  np.array([Patch[:,0],Patch[:,1]]).T
Q = np.array([Patch[:,0],Patch[:,4]]).T

plt.scatter(P[:,0],P[:,1])
plt.scatter(Q[:,0],Q[:,1])
plt.axis('equal')
plt.axis('off')

In [ ]:
barc = mtd.calc_cross_barcodes(P, Q, batch_size1 = 100, batch_size2 = 1000, is_plot = True)

In [ ]:
plt.figure(figsize=(10,10))
ax = plt.axes()


d = barc[1][:,1]
b = barc[1][:,0]

mx=np.max(d)
mx = np.log(1.1*mx)
mn1 = np.min(b)
mn = np.log(mn1)
ax.fill([mn,mx,mx],[mn,mn,mx],facecolor='black',edgecolor='gray',alpha=0.1)

plt.scatter(np.log(b),np.log(d),facecolor='black', edgecolor='white',s=50, alpha=0.6*d/np.max(d)+0.2)

m = np.log(slope(b,d,alpha,1))
ax.plot([mn,mx],[mn+m,mx+m],':',c='blue')


ax.set_xlabel('Log(Birth)',fontsize=20)
ax.set_ylabel('Log(Death)',fontsize=20)
ax.set(xlim=(mn, mx), ylim=(mn, mx))
ax.tick_params(axis='x', labelsize=15)
ax.tick_params(axis='y', labelsize=15)
ax.set_title("H1 with noise")

ax.set_aspect('equal')
plt.plot()

In [ ]:
d_b = d/b
X = process(d_b)
res, flags, _ = test(X, alpha)

In [ ]:
res

In [ ]:
print(f"before : {mtd.get_score(barc, 1, 'sum_length')}")

In [ ]:
barc_denoised = barc.copy()
barc_denoised[1] = barc_denoised[1][res]

In [ ]:
mtd.get_score(barc_denoised, 1, 'sum_length')

## MNIST

In [ ]:
from sklearn.datasets import fetch_openml
from PIL import Image

In [ ]:
# Load data from https://www.openml.org/d/554
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)

In [ ]:
img = Image.fromarray(255 - np.uint8(X[47].reshape((28, 28))), 'L')

In [ ]:
img

In [ ]:
images = []

for idx in range(len(y)):
    if y[idx] == '5':
        img = Image.fromarray(np.uint8(X[idx].reshape((28, 28))), 'L')
        images.append(img)

In [ ]:
#Create point clouds of "5"s and flipped "5"s
def get_dataset(angle):
    fives = []

    for i, elem in enumerate(y):
        if elem == '5':
            A = np.zeros((40, 40))
            A[6:34, 6:34] = X[i].reshape((28, 28))
            img = Image.fromarray(np.uint8(A), 'L')
            
            if angle > 0:
                img = img.transpose(Image.FLIP_TOP_BOTTOM)
            
            fives.append(np.asarray(img).flatten())
    
    return np.array(fives)

#Create point clouds of "5"s and  "4"s
def get_dataset_diff(number):
    images = []

    for i, elem in enumerate(y):
        if elem == number:
            A = np.zeros((40, 40))
            A[6:34, 6:34] = X[i].reshape((28, 28))
            img = Image.fromarray(np.uint8(A), 'L')
            
            images.append(np.asarray(img).flatten())
    
    return np.array(images)

In [ ]:
# clouds = []

# for angle in [0, 1]:
#     clouds.append(get_dataset(angle))

In [ ]:
clouds = []

for number in ["5", "3"]:
    clouds.append(get_dataset_diff(number))

In [ ]:
for cloud in clouds:
    print(cloud.shape)

In [ ]:
res1 = []
trials = 20
numbers = ["5", "4", "3"]

for i in range(1, len(clouds)):
    print(f"comparing {numbers[i]}'s with 5's")
    np.random.seed(7)
    barcs = [mtd.calc_cross_barcodes(clouds[i], clouds[0], batch_size1 = 100, batch_size2 = 1000) for _ in range(trials)]
    res1.append(barcs)

In [ ]:

def get_scores(res, args_dict, trials = 10):

    scores = []

    for i in range(len(res)): 
        asum = []
        
        for exp_id, elem in enumerate(res[i]):
            asum.append(mtd.get_score(elem, **args_dict))

        scores.append(sum(asum) / len(res[i]))

    return scores

In [ ]:
scores = get_scores(res1, {'h_idx' : 1, 'kind' : 'sum_length'})

In [ ]:
for s in scores:
    print(s)

In [ ]:
fig, axes = plt.subplots(nrows=trials, ncols=1, figsize=(200,200))
i=-1
for barc in res1[1]:
    i = i + 1
    ax = axes[i]
    
    
    d = barc[1][:,1].astype('float64')
    b = barc[1][:,0].astype('float64')
    
    mx=np.max(d)
    mx = np.log(1.1*mx)
    mn1 = np.min(b)
    mn = np.log(mn1)
    ax.fill([mn,mx,mx],[mn,mn,mx],facecolor='black',edgecolor='gray',alpha=0.1)
    
    ax.scatter(np.log(b),np.log(d),facecolor='black', edgecolor='white',s=5, alpha=0.6*d/np.max(d)+0.2)
    
    m = np.log(slope(b,d,alpha,1))
    ax.plot([mn,mx],[mn+m,mx+m],':',c='blue')
    
    
    ax.set_xlabel('Log(Birth)',fontsize=5)
    ax.set_ylabel('Log(Death)',fontsize=5)
    ax.set(xlim=(mn, mx), ylim=(mn, mx))
    ax.tick_params(axis='x', labelsize=3)
    ax.tick_params(axis='y', labelsize=3)
    
    ax.set_aspect('equal')
plt.plot()

## Let's check the main hypotesis about univrsal distribution

In [ ]:
import seaborn as sns
import scipy.stats as st
import scipy.special as sp
import statsmodels.api as sm

In [ ]:
rv=st.gumbel_l()

COLS = sns.color_palette()

In [ ]:
np.random.seed(7)

P = np.loadtxt(f'{ROOT}annulus.csv')
Q = np.loadtxt(f'{ROOT}annulus1.csv') + np.array([1,0])

In [ ]:
plt.scatter(P[:,0],P[:,1])
plt.scatter(Q[:,0],Q[:,1])

In [ ]:
barc = mtd.calc_cross_barcodes(P, Q, batch_size1 = 100, batch_size2 = 1000, is_plot = True)

In [ ]:
res1 = []
trials = 200

np.random.seed(7)
barcs = [mtd.calc_cross_barcodes(P, Q, batch_size1 = 100, batch_size2 = 1000) for _ in range(trials)]
res1.append(barcs)

In [ ]:
b = np.concatenate([bar[1][:,0].astype('float64') for bar in res1[0]], axis = 0)
d = np.concatenate([bar[1][:,1].astype('float64') for bar in res1[0]], axis = 0)

m = np.log(slope(b,d,alpha,1))
print(f"current slope coef: {m}")

fig, axes = plt.subplots(nrows=20, ncols=1, figsize=(200,200))

i=-1
for barc in res1[0][:20]:
    i = i + 1
    ax = axes[i]
    
    
    d = barc[1][:,1].astype('float64')
    b = barc[1][:,0].astype('float64')
    
    mx=np.max(d)
    mx = np.log(1.1*mx)
    mn1 = np.min(b)
    mn = np.log(mn1)
    ax.fill([mn,mx,mx],[mn,mn,mx],facecolor='black',edgecolor='gray',alpha=0.1)
    
    ax.scatter(np.log(b),np.log(d),facecolor='black', edgecolor='white',s=5, alpha=0.6*d/np.max(d)+0.2)
    
    ax.plot([mn,mx],[mn+m,mx+m],':',c='blue')
    
    
    ax.set_xlabel('Log(Birth)',fontsize=8)
    ax.set_ylabel('Log(Death)',fontsize=8)
    ax.set(xlim=(mn, mx), ylim=(mn, mx))
    ax.tick_params(axis='x', labelsize=8)
    ax.tick_params(axis='y', labelsize=8)
    
    ax.set_aspect('equal')
plt.plot()

In [ ]:
samples = np.array([[0],[0]]).reshape((1,2))

for i in range(trials):
    samples = np.append(samples, res1[0][i][1], axis = 0)

samples = samples[1:]
samples.shape

In [ ]:
birth = samples[:, 0]
death = samples[:, 1]
pall = death/birth

In [ ]:
pall = np.log(pall)

In [ ]:
pall = np.log(pall)
pall = pall - pall.mean() + rv.mean()

In [ ]:
lnwidth = 3
xl=(-6,4)
xlqq=(-10,4)
lnwidth=3
fnsize_title=20 
fnsize_legend = 14 
fnsize_tick = 14

if xl!=None:
    pall_cdf = np.append(pall, xl)
else:
    pall_cdf = pall

adj = [2,1.5]
plt.figure(figsize=(18,6))

ax = plt.subplot(1, 3, 1)
sns.ecdfplot(pall, ax=ax, linewidth=lnwidth, color = COLS[1])

ax = plt.subplot(1,3,2)

sns.kdeplot(pall, ax=ax, linewidth=lnwidth)#/prms[4]*4)
# x, y = NaiveKDE(kernel='cosine', bw='ISJ').fit(pall).evaluate()
# plt.plot(x,y)

ax = plt.subplot(1,3,3)
pp_y = sm.ProbPlot(pall, dist=rv)

pp_y.qqplot(ax=ax, markersize=5)



ax = plt.subplot(1, 3, 1)
ax.set(ylim=(-0.05, 1.05))
if xl==None:
    xl = ax.get_xlim()

x = np.linspace(xl[0],xl[1],1000)
ax.plot(x, rv.cdf(x), 'k:', linewidth=5, label='LGumbel')
ax.tick_params(axis='x', labelsize=fnsize_tick)
ax.tick_params(axis='y', labelsize=fnsize_tick)

ax.set_xlim(xl)
ax.set_ylabel(None)

ax.set_title('CDF', fontsize=fnsize_title)

ax.legend(loc='upper left', fontsize=fnsize_legend)

ax = plt.subplot(1,3,2)
ax.plot(x, rv.pdf(x), 'k:', linewidth=5, label='LGumbel')
ax.set_xlim(xl)
yl = ax.get_ylim()
ax.set_ylim([-0.05, yl[1]])
ax.set_ylabel(None)
ax.set_title('PDF', fontsize=fnsize_title)
ax.tick_params(axis='x', labelsize=fnsize_tick)
ax.tick_params(axis='y', labelsize=fnsize_tick)

if xlqq==None:
    xlqq = xl
ax = plt.subplot(1, 3, 3)
ax.plot(xlqq, xlqq,'--k', linewidth=3)
ax.set_xlim(xlqq)
ax.set_ylim(xlqq)
ax.set_title('QQ-Plot', fontsize=fnsize_title)
# ax.set_xlabel('Log-Exponential Quantiles', fontsize=fnsize)
# ax.set_ylabel('Sample Quantiles', fontsize=fnsize)
ax.set_xlabel(None)
ax.set_ylabel(None)
ax.tick_params(axis='x', labelsize=fnsize_tick)
ax.tick_params(axis='y', labelsize=fnsize_tick)

plt.tight_layout()

It is L_Gumbel, maybe

Let's check another domain

In [ ]:
res1 = []
trials = 100
numbers = ["5", "3"]

for i in range(1, len(clouds)):
    print(f"comparing {numbers[i]}'s with 5's")
    np.random.seed(7)
    barcs = [mtd.calc_cross_barcodes(clouds[i], clouds[0], batch_size1 = 100, batch_size2 = 1000) for _ in range(trials)]
    res1.append(barcs)

In [ ]:
samples = np.array([[0],[0]]).reshape((1,2))

for i in range(trials):
    samples = np.append(samples, res1[0][i][1], axis = 0)

samples = samples[1:]
samples.shape

birth = samples[:, 0]
death = samples[:, 1]
pall = np.array(death/birth, dtype = float)

pall = np.log(pall)

pall = np.log(pall)
pall = pall - pall.mean() + rv.mean()

lnwidth = 3
xl=(-6,4)
xlqq=(-10,4)
lnwidth=3
fnsize_title=20 
fnsize_legend = 14 
fnsize_tick = 14

if xl!=None:
    pall_cdf = np.append(pall, xl)
else:
    pall_cdf = pall

adj = [2,1.5]
plt.figure(figsize=(18,6))

ax = plt.subplot(1, 3, 1)
sns.ecdfplot(pall, ax=ax, linewidth=lnwidth, color = COLS[1])

ax = plt.subplot(1,3,2)

sns.kdeplot(pall, ax=ax, linewidth=lnwidth, clip=xl)#/prms[4]*4)
# x, y = NaiveKDE(kernel='cosine', bw='ISJ').fit(pall).evaluate()
# plt.plot(x,y)

ax = plt.subplot(1,3,3)
pp_y = sm.ProbPlot(pall, dist=rv)

pp_y.qqplot(ax=ax, markersize=5)



ax = plt.subplot(1, 3, 1)
ax.set(ylim=(-0.05, 1.05))
if xl==None:
    xl = ax.get_xlim()

x = np.linspace(xl[0],xl[1],1000)
ax.plot(x, rv.cdf(x), 'k:', linewidth=5, label='LGumbel')
ax.tick_params(axis='x', labelsize=fnsize_tick)
ax.tick_params(axis='y', labelsize=fnsize_tick)

ax.set_xlim(xl)
ax.set_ylabel(None)

ax.set_title('CDF', fontsize=fnsize_title)

ax.legend(loc='upper left', fontsize=fnsize_legend)

ax = plt.subplot(1,3,2)
ax.plot(x, rv.pdf(x), 'k:', linewidth=5, label='LGumbel')
ax.set_xlim(xl)
yl = ax.get_ylim()
ax.set_ylim([-0.05, yl[1]])
ax.set_ylabel(None)
ax.set_title('PDF', fontsize=fnsize_title)
ax.tick_params(axis='x', labelsize=fnsize_tick)
ax.tick_params(axis='y', labelsize=fnsize_tick)

if xlqq==None:
    xlqq = xl
ax = plt.subplot(1, 3, 3)
ax.plot(xlqq, xlqq,'--k', linewidth=3)
ax.set_xlim(xlqq)
ax.set_ylim(xlqq)
ax.set_title('QQ-Plot', fontsize=fnsize_title)
# ax.set_xlabel('Log-Exponential Quantiles', fontsize=fnsize)
# ax.set_ylabel('Sample Quantiles', fontsize=fnsize)
ax.set_xlabel(None)
ax.set_ylabel(None)
ax.tick_params(axis='x', labelsize=fnsize_tick)
ax.tick_params(axis='y', labelsize=fnsize_tick)

plt.tight_layout()

---

## COIL-20

https://github.com/danchern97/RTD_AE/tree/main

In [ ]:
import os
from tqdm import tqdm
from PIL import Image
import numpy as np
import torch

filenames = os.listdir("COIL-20_data/coil-20-proc/")
dirname = "COIL-20_data/coil-20-proc/"

labels = []
data = []
for file in tqdm(filenames):
    img = Image.open(dirname + file)
    objId, imgId = file.split('__')
    imgId = int(imgId[:-4])
    objId = int(objId[3:])
    data.append(np.array(img))
    labels.append(objId)
data = np.asarray(data, dtype="float")
labels = np.asarray(labels)

In [ ]:
from scipy.spatial.distance import cdist
from scipy.spatial import distance_matrix
class FurthestScaler:
    def __init__(self, p=2): # approximate
        self.is_fitted = False
        self.p = p
        
    def fit(self, data):
        self.furthest = self._furthest_distance(data)
        self.is_fitted = True
        
    def transform(self, data):
        if not self.is_fitted:
            raise NotFittedError
        return data / self.furthest
    
    def fit_transform(self, data):
        self.fit(data)
        return self.transform(data)

    def _furthest_distance(self, points, sample_frac=0.0):
        # exact solution, very computationaly expesive
        # hull = ConvexHull(points) 
        # hullpoints = points[hull.vertices,:]
        # hdist = distance_matrix(hullpoints, hullpoints, p=self.p)
        # approximation: upper bound
        # pick random point and compute distances to all of the points
        # diameter min: max(distances), diameter max (triangle inequality): 2 max(distances)
        if len(points.shape) > 2:
            points = points.reshape(points.shape[0],-1)
        idx = np.random.choice(np.arange(len(points)), size=1)
        hdist = distance_matrix(points[idx], points, p=self.p)
        return 0.1*hdist.max() # upper bound

In [ ]:
scaler = FurthestScaler()
data = torch.tensor(data).flatten(start_dim=1).numpy()
data = scaler.fit_transform(data)

In [ ]:
classes = list(set(labels))

clouds = []
for i in classes:
    clouds.append(data[labels==i])

In [ ]:
barcs_of_same_clouds = []
n = 100
for i in range(len(clouds)):
    barcs = [mtd.calc_cross_barcodes(clouds[i], clouds[i], batch_size1 = 50, batch_size2 = 60, pdist_device = "cuda",is_plot = False) for _ in range(n)]
    barcs_of_same_clouds.append(barcs)
    print(f"{classes[i]} is done !!!")

In [ ]:
barcs_of_diff_clouds_fix = {}
n = 100
for i in range(len(clouds)):
    for j in range(len(clouds)):
        if i!=j:
            barcs = [mtd.calc_cross_barcodes(clouds[i], clouds[j], batch_size1 = 50, batch_size2 = 60, pdist_device = "cuda",is_plot = False) for _ in range(n)]
            barcs_of_diff_clouds_fix[f"{classes[i]} vs {classes[j]}"] = barcs
            print(f"{classes[i]} vs {classes[j]} is done !!!")

---

In [ ]:
b = np.concatenate([bar[1][:,0].astype('float64') for bar in barcs_of_diff_clouds_fix["18 vs 7"]], axis = 0)
d = np.concatenate([bar[1][:,1].astype('float64') for bar in barcs_of_diff_clouds_fix["18 vs 7"]], axis = 0)

m = np.log(slope(b,d,alpha,1))
print(f"current slope coef: {m}")

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(16,8))


# d = barc[1][:,1].astype('float64')
# b = barc[1][:,0].astype('float64')

mx=np.max(d)
mx = np.log(1.1*mx)
mn1 = np.min(b)
mn = np.log(mn1)
ax.fill([mn,mx,mx],[mn,mn,mx],facecolor='black',edgecolor='gray',alpha=0.1)

ax.scatter(np.log(b),np.log(d),facecolor='black', edgecolor='white',s=8, alpha=0.7*d/np.max(d)+0.2)

ax.plot([mn,mx],[mn+m,mx+m],':',c='blue')


ax.set_xlabel('Log(Birth)',fontsize=8)
ax.set_ylabel('Log(Death)',fontsize=8)
ax.set(xlim=(mn, mx), ylim=(mn, mx))
ax.tick_params(axis='x', labelsize=8)
ax.tick_params(axis='y', labelsize=8)

ax.set_aspect('equal')
plt.plot()

In [ ]:
samples = np.array([[0],[0]]).reshape((1,2))

for i in range(n):
    samples = np.append(samples, barcs_of_diff_clouds_fix["18 vs 7"][i][1], axis = 0)

samples = samples[1:]
print(samples.shape)

birth = samples[:, 0]
death = samples[:, 1]
pall = np.array(death/birth, dtype = float)

pall = np.log(pall)

pall = np.log(pall)
pall = pall - pall.mean() + rv.mean()

lnwidth = 3
xl=(-6,4)
xlqq=(-10,4)
lnwidth=3
fnsize_title=20 
fnsize_legend = 14 
fnsize_tick = 14

if xl!=None:
    pall_cdf = np.append(pall, xl)
else:
    pall_cdf = pall

adj = [2,1.5]
plt.figure(figsize=(18,6))

ax = plt.subplot(1, 3, 1)
sns.ecdfplot(pall, ax=ax, linewidth=lnwidth, color = COLS[1])

ax = plt.subplot(1,3,2)

sns.kdeplot(pall, ax=ax, linewidth=lnwidth, clip=xl)#/prms[4]*4)
# x, y = NaiveKDE(kernel='cosine', bw='ISJ').fit(pall).evaluate()
# plt.plot(x,y)

ax = plt.subplot(1,3,3)
pp_y = sm.ProbPlot(pall, dist=rv)

pp_y.qqplot(ax=ax, markersize=5)



ax = plt.subplot(1, 3, 1)
ax.set(ylim=(-0.05, 1.05))
if xl==None:
    xl = ax.get_xlim()

x = np.linspace(xl[0],xl[1],1000)
ax.plot(x, rv.cdf(x), 'k:', linewidth=5, label='LGumbel')
ax.tick_params(axis='x', labelsize=fnsize_tick)
ax.tick_params(axis='y', labelsize=fnsize_tick)

ax.set_xlim(xl)
ax.set_ylabel(None)

ax.set_title('CDF', fontsize=fnsize_title)

ax.legend(loc='upper left', fontsize=fnsize_legend)

ax = plt.subplot(1,3,2)
ax.plot(x, rv.pdf(x), 'k:', linewidth=5, label='LGumbel')
ax.set_xlim(xl)
yl = ax.get_ylim()
ax.set_ylim([-0.05, yl[1]])
ax.set_ylabel(None)
ax.set_title('PDF', fontsize=fnsize_title)
ax.tick_params(axis='x', labelsize=fnsize_tick)
ax.tick_params(axis='y', labelsize=fnsize_tick)

if xlqq==None:
    xlqq = xl
ax = plt.subplot(1, 3, 3)
ax.plot(xlqq, xlqq,'--k', linewidth=3)
ax.set_xlim(xlqq)
ax.set_ylim(xlqq)
ax.set_title('QQ-Plot', fontsize=fnsize_title)
# ax.set_xlabel('Log-Exponential Quantiles', fontsize=fnsize)
# ax.set_ylabel('Sample Quantiles', fontsize=fnsize)
ax.set_xlabel(None)
ax.set_ylabel(None)
ax.tick_params(axis='x', labelsize=fnsize_tick)
ax.tick_params(axis='y', labelsize=fnsize_tick)

plt.tight_layout()